In [4]:
import torch

# Câu lệnh tự động chọn thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Đang sử dụng thiết bị: {device}")


Đang sử dụng thiết bị: cpu


# Notebook 00 — Master ingestion & batch assignment (Colab drive-shadow + resumable standardize + versioned raw upload)

Workflow đúng của notebook này:

```text
Google Drive folder của BTC / nguồn tổ chức
→ drive-shadow copy sang folder Drive của team/bạn
→ Colab đọc folder Drive local đó qua /content/drive/...
→ standardize-archives --resume bằng local temp `/content/aic_scratch`, output thành raw_videos/metadata trên Drive
→ ingest local trực tiếp từ folder standardize
→ assign-batches tạo batch_manifest.csv + batch_*.txt
→ upload processed artifacts nhỏ lên Hugging Face processed repo
→ optional upload standardized raw_videos + metadata lên Hugging Face raw repo theo version prefix
→ cập nhật media_store_manifest.parquet trong processed repo để trỏ tới HF raw canonical
```

Thiết kế repo:

```text
AIC2026_processed = release/control plane
AIC2026_raw       = versioned canonical raw store
```

Raw repo dùng version prefix:

```text
AIC2026_raw/
└── canonical_dataset_v001/
    ├── raw_videos/
    ├── metadata/
    └── manifests/
        ├── canonical_file_manifest.jsonl
        └── canonical_import_report.json
```

Điểm quan trọng:

- `drive_target_id` phải là **folder ID đúng của folder local `archive_source_dir`**. Ví dụ nếu `archive_source_dir = /content/drive/MyDrive/AIC2026/raw_dataset`, thì `drive_target_id` phải chính là folder ID của `raw_dataset` trên Google Drive.
- `temp_extract_dir` **phải nằm trên local runtime** (`/content/aic_scratch`) để tránh ghi temp qua Google DriveFS. Không đặt temp trong `/content/drive`.
- BƯỚC 2 dùng `--resume`, `standardize_progress.jsonl`, local temp cleanup, và DriveFS throttle. CLI cần agent hỗ trợ `--min-free-gb`, `--drive-sync-sleep-seconds`, `--cleanup-every-files`, `--cleanup-every-gb`.
- BƯỚC 3 dùng ingest local. CLI cần hỗ trợ `--source-uri`, `--max-workers`, và `--pairing-policy video-primary`.
- BƯỚC 7 dùng command mới `upload-standardized-raw`. CLI cần được agent thêm command này trước khi bật `run_upload_standardized_raw=True`.


In [5]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass

@dataclass
class WorkflowConfig:
    # 1. Hugging Face repos
    # Raw repo: versioned canonical raw store.
    hf_canonical_repo: str = "1thesudden/AIC2026_raw"
    raw_import_id: str = "canonical_dataset_v002"

    # Processed repo: release/control plane.
    hf_release_repo: str = "1thesudden/AIC2026_processed"
    release_id: str = "canonical_release_v002"

    # Raw upload là bước nặng, mặc định tắt.
    # Bật True sau khi processed artifacts đã pass.
    run_upload_standardized_raw: bool = True

    # 2. Google Drive IDs
    # Workflow bắt buộc: copy source Drive folder của BTC sang folder Drive của team/bạn trước.
    drive_source_id: str = "1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK" # https://drive.google.com/drive/folders/1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK?usp=drive_link
    drive_target_id: str = "1roEruPwNWxBt8lJUGib4_M_ehl9Shj64" # https://drive.google.com/drive/folders/1roEruPwNWxBt8lJUGib4_M_ehl9Shj64?usp=drive_link
    run_drive_shadow: bool = True

    # 3. Local paths nhìn từ Colab sau khi mount Google Drive.
    # RẤT QUAN TRỌNG: archive_source_dir phải là local mount path tương ứng với drive_target_id.
    # drive-shadow copy vào drive_target_id; standardize đọc từ archive_source_dir.
    archive_source_dir: str = "/content/drive/MyDrive/AIC2026/raw_dataset"
    archive_target_dir: str = "/content/drive/MyDrive/AIC2026/standardize"

    # Temp phải nằm trên local runtime để giảm ghi/cache qua Google DriveFS.
    # Package phải cleanup temp sau từng member/source và throttle khi disk thấp.
    # Không đặt temp_extract_dir trong /content/drive cho dataset lớn.
    temp_extract_dir: str = "/content/temp_extract"

    # 4. Repo code
    github_repo_url: str = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
    github_branch: str = "system1-optimized"
    repo_dir_name: str = "Multimodal-Agentic-Retrieval-Engine"

    # 5. Execution Options
    execution_mode: str = "bronze_fast"
    num_batches: int = 10
    run_standardize_archives: bool = True
    run_ingest: bool = True
    run_assign_batches: bool = True

    # Upload processed artifacts nhỏ sau khi chia batch.
    run_upload_processed: bool = True

    def __post_init__(self):
        self.env = "colab" if "google.colab" in sys.modules else "local"
        self.workspace = Path("/content") if self.env == "colab" else Path.cwd()
        os.environ["AIC_RELEASE_ID"] = self.release_id
        os.environ["AIC_HF_REPO_ID"] = self.hf_release_repo

config = WorkflowConfig()

print("Môi trường:", config.env)
print("Workspace:", config.workspace)
print("Release ID:", config.release_id)
print("HF processed repo:", config.hf_release_repo)
print("HF raw repo:", config.hf_canonical_repo)
print("Raw import ID:", config.raw_import_id)
print("Run upload standardized raw:", config.run_upload_standardized_raw)
print("Run drive shadow:", config.run_drive_shadow)
print("Drive source ID:", config.drive_source_id)
print("Drive target ID:", config.drive_target_id)
print("Archive source dir dùng để standardize:", config.archive_source_dir)
print("Archive target dir:", config.archive_target_dir)
print("Temp extract dir:", config.temp_extract_dir)


Môi trường: colab
Workspace: /content
Release ID: canonical_release_v002
HF processed repo: 1thesudden/AIC2026_processed
HF raw repo: 1thesudden/AIC2026_raw
Raw import ID: canonical_dataset_v002
Run upload standardized raw: True
Run drive shadow: True
Drive source ID: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
Drive target ID: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
Archive source dir dùng để standardize: /content/drive/MyDrive/AIC2026/raw_dataset
Archive target dir: /content/drive/MyDrive/AIC2026/standardize
Temp extract dir: /content/temp_extract


In [6]:
import shutil
import subprocess
from pathlib import Path

# 1. Mount Drive và lấy HF token.
if config.env == "colab":
    from google.colab import userdata, drive, auth
    hf_token = userdata.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN trong Colab Secrets. Hãy thêm HF_TOKEN trước khi chạy notebook.")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["AIC_HF_TOKEN"] = hf_token

    drive.mount("/content/drive", force_remount=False)
    auth.authenticate_user()
else:
    if not os.environ.get("HF_TOKEN") and not os.environ.get("AIC_HF_TOKEN"):
        print("Cảnh báo: chưa thấy HF_TOKEN/AIC_HF_TOKEN trong environment.")

# 2. Clone repo.
repo_dir = config.workspace / config.repo_dir_name
if repo_dir.exists():
    print(f"Xóa repo cũ: {repo_dir}")
    shutil.rmtree(repo_dir)

print("Clone repo:", config.github_repo_url)
subprocess.run([
    "git", "clone",
    "--branch", config.github_branch,
    config.github_repo_url,
    str(repo_dir),
], check=True)

# 3. Tự tìm project root đúng.
candidate_roots = [repo_dir, repo_dir / "system1"]
project_root = None
for p in candidate_roots:
    if (p / "pyproject.toml").exists() and (p / "src" / "system1").exists():
        project_root = p
        break

if project_root is None:
    print("Repo tree sau khi clone:")
    subprocess.run(["bash", "-lc", f"find {repo_dir} -maxdepth 3 -type f | sort | head -200"])
    raise RuntimeError("Không tìm thấy project root có pyproject.toml và src/system1.")

print("Project root:", project_root)

# 4. Cài dependencies tối thiểu và package editable.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gdown", "pydrive2", "huggingface_hub", "pyarrow", "pandas",
    "-e", str(project_root),
], check=True)

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Cài package xong.")


Mounted at /content/drive
Clone repo: https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git
Project root: /content/Multimodal-Agentic-Retrieval-Engine/system1
Cài package xong.


In [7]:
def run_cli(args, *, check=True):
    import os
    import subprocess
    import sys
    from pathlib import Path

    def find_repo_root():
        candidates = [
            globals().get("SYSTEM1_ROOT"),
            globals().get("REPO_ROOT"),
            Path.cwd(),
            Path("/content/Multimodal-Agentic-Retrieval-Engine"),
        ]

        for candidate in candidates:
            if candidate is None:
                continue
            path = Path(candidate).expanduser().resolve()

            # Nếu truyền vào system1/ thì lấy parent repo.
            if path.name == "system1" and (path / "src" / "system1").exists():
                return path

            # Repo root có folder system1/src/system1.
            if (path / "system1" / "src" / "system1").exists():
                return path

        raise RuntimeError(
            "Không tìm thấy repo root. Hãy chạy cell clone/setup repo trước, "
            "hoặc kiểm tra repo có nằm ở /content/Multimodal-Agentic-Retrieval-Engine không."
        )

    cli_cwd = find_repo_root()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    system1_src = cli_cwd / "system1" / "src"
    env["PYTHONPATH"] = str(system1_src) + os.pathsep + env.get("PYTHONPATH", "")

    cmd = [sys.executable, "-m", "system1.cli", *args]

    print("\n" + "=" * 100)
    print("CWD:", cli_cwd)
    print("PYTHONPATH prefix:", system1_src)
    print("RUN:", " ".join(cmd))
    print("=" * 100)

    process = subprocess.Popen(
        cmd,
        cwd=str(cli_cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    returncode = process.wait()
    output = "".join(lines)

    if check and returncode != 0:
        raise RuntimeError(
            f"CLI failed with exit code {returncode}: {' '.join(args)}"
        )

    return output

In [8]:
from system1.runtime.environment import resolve_runtime_paths
from pathlib import Path

runtime_paths = resolve_runtime_paths(output_root=config.workspace / "output")
output_base = runtime_paths.output_root
output_base.mkdir(parents=True, exist_ok=True)

print("Runtime paths:")
print("- environment:", runtime_paths.environment)
print("- workspace_root:", runtime_paths.workspace_root)
print("- input_root:", runtime_paths.input_root)
print("- output_root:", runtime_paths.output_root)
print("- artifact_root:", runtime_paths.artifact_root)
print("- release_id:", runtime_paths.release_id)

archive_source = Path(config.archive_source_dir)
archive_target = Path(config.archive_target_dir)
temp_extract = Path(config.temp_extract_dir)

print("Workflow check:")
print("- drive-shadow sẽ copy vào drive_target_id:", config.drive_target_id)
print("- standardize sẽ đọc local archive_source_dir:", archive_source)
print("- Hai giá trị này phải trỏ tới cùng một folder Google Drive.")

print("Trạng thái archive_source_dir trước drive-shadow:")
print("- exists:", archive_source.exists())
if archive_source.exists():
    visible_items = list(archive_source.rglob("*"))
    print("- visible item count before shadow:", len(visible_items))
    for p in visible_items[:30]:
        print(" ", p)
else:
    print("- Chưa thấy folder local. Notebook vẫn sẽ chạy drive-shadow trước, sau đó remount Drive và kiểm tra lại.")


Runtime paths:
- environment: colab
- workspace_root: /content
- input_root: /content/input
- output_root: /content/output
- artifact_root: /content/system1_artifacts
- release_id: canonical_release_v002
Workflow check:
- drive-shadow sẽ copy vào drive_target_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- standardize sẽ đọc local archive_source_dir: /content/drive/MyDrive/AIC2026/raw_dataset
- Hai giá trị này phải trỏ tới cùng một folder Google Drive.
Trạng thái archive_source_dir trước drive-shadow:
- exists: True
- visible item count before shadow: 15
  /content/drive/MyDrive/AIC2026/raw_dataset/media-info-aic25-b1.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L24_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L25_a1.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_a.zip
  /co

In [9]:
from pathlib import Path
import json
import time
from google.colab import drive

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"
archive_source = Path(config.archive_source_dir)

if config.run_drive_shadow:
    if not config.drive_source_id or not config.drive_target_id:
        raise RuntimeError("run_drive_shadow=True nhưng thiếu drive_source_id hoặc drive_target_id.")

    print("Bắt đầu drive-shadow:")
    print("- source_folder_id:", config.drive_source_id)
    print("- dest_folder_id:", config.drive_target_id)
    print("- report_path:", drive_shadow_report_path)

    run_cli([
        "drive-shadow",
        "--source-folder-id", config.drive_source_id,
        "--dest-folder-id", config.drive_target_id,
        "--report-path", str(drive_shadow_report_path),
    ])
else:
    raise RuntimeError(
        "Workflow hiện tại yêu cầu chạy drive-shadow trước. "
        "Hãy để config.run_drive_shadow = True."
    )

Bắt đầu drive-shadow:
- source_folder_id: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
- dest_folder_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- report_path: /content/drive_shadow_report.json

CWD: /content/Multimodal-Agentic-Retrieval-Engine
PYTHONPATH prefix: /content/Multimodal-Agentic-Retrieval-Engine/system1/src
RUN: /usr/bin/python3 -m system1.cli drive-shadow --source-folder-id 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK --dest-folder-id 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64 --report-path /content/drive_shadow_report.json
[drive-shadow] source_folder_id=1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK dest_folder_id=1roEruPwNWxBt8lJUGib4_M_ehl9Shj64 report_path=/content/drive_shadow_report.json
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
httplib2 transport does not support per-request timeout. Set the timeout when constructing the httplib2.Http instance.
[drive-shadow] source folder: video_batch_1 (1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK)
[drive-shadow] 

In [10]:
from pathlib import Path
import json

drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Kiểm tra drive-shadow report:")
print("- path:", drive_shadow_report_path)
print("- exists:", drive_shadow_report_path.exists())

drive_shadow_report = None
copied_files = 0
created_folders = 0
skipped_existing = 0
skipped_google_apps = 0
error_count = None

if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

    copied_files = int(drive_shadow_report.get("copied_files", 0))
    created_folders = int(drive_shadow_report.get("created_folders", 0))
    skipped_existing = int(drive_shadow_report.get("skipped_existing", 0))
    skipped_google_apps = int(drive_shadow_report.get("skipped_google_apps", 0))
    error_count = int(drive_shadow_report.get("error_count", 0))

    print("Drive shadow summary:")
    print("- status:", drive_shadow_report.get("status"))
    print("- source_folder_id:", drive_shadow_report.get("source_folder_id"))
    print("- dest_folder_id:", drive_shadow_report.get("dest_folder_id"))
    print("- source_folder_name:", drive_shadow_report.get("source_folder_name"))
    print("- dest_folder_name:", drive_shadow_report.get("dest_folder_name"))
    print("- copied_files:", copied_files)
    print("- created_folders:", created_folders)
    print("- skipped_existing:", skipped_existing)
    print("- skipped_google_apps:", skipped_google_apps)
    print("- error_count:", error_count)

    if error_count and error_count > 0:
        print("\nMột số item lỗi đầu tiên:")
        failed_items = [
            item for item in drive_shadow_report.get("items", [])
            if item.get("status") == "failed"
        ]
        for item in failed_items[:20]:
            print(item)

        raise RuntimeError(
            "drive-shadow có error_count > 0. "
            "Không chạy standardize khi copy Drive chưa sạch lỗi."
        )

    if copied_files + created_folders + skipped_existing + skipped_google_apps == 0:
        raise RuntimeError(
            "drive-shadow chạy xong nhưng report cho thấy không copy/tạo/skip bất kỳ item nào. "
            "Kiểm tra source folder hoặc quyền Drive."
        )
else:
    raise FileNotFoundError(
        f"Không thấy drive-shadow report: {drive_shadow_report_path}. "
        "CLI drive-shadow cần tạo report sau khi chạy."
    )

Kiểm tra drive-shadow report:
- path: /content/drive_shadow_report.json
- exists: True
Drive shadow summary:
- status: pass
- source_folder_id: 1o4Rrfmu5ZHhpt_y5UjbzPZSzJsHO2klK
- dest_folder_id: 1roEruPwNWxBt8lJUGib4_M_ehl9Shj64
- source_folder_name: video_batch_1
- dest_folder_name: raw_dataset
- copied_files: 0
- created_folders: 0
- skipped_existing: 15
- skipped_google_apps: 0
- error_count: 0


In [11]:
from pathlib import Path
import json
import time
from google.colab import drive

archive_source = Path(config.archive_source_dir)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

print("Remount Google Drive để Colab nhìn thấy dữ liệu mới...")

try:
    drive.flush_and_unmount()
except Exception as exc:
    print("flush_and_unmount warning:", exc)

time.sleep(5)
drive.mount("/content/drive", force_remount=True)

print("Kiểm tra local archive_source_dir sau drive-shadow:")
print("- archive_source_dir:", archive_source)
print("- exists:", archive_source.exists())

if not archive_source.exists():
    raise RuntimeError(
        "Không thấy archive_source_dir sau drive-shadow. "
        "archive_source_dir phải là local path tương ứng với drive_target_id."
    )

if not drive_shadow_report_path.exists():
    raise RuntimeError(f"Không thấy drive-shadow report để đối chiếu: {drive_shadow_report_path}")

drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))

# Chỉ các file thường đã copied/skipped_existing mới cần xuất hiện trong Drive mount.
# skipped_google_apps không phải file local; created_folders không tính vào file count.
expected_relative_files = sorted({
    str(item.get("path", "")).strip("/")
    for item in drive_shadow_report.get("items", [])
    if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
})

expected_file_count_from_report = int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
expected_file_count = len(expected_relative_files) or expected_file_count_from_report

print("Drive mount sync expectation:")
print("- expected_file_count:", expected_file_count)
print("- expected_relative_files sample:", expected_relative_files[:20])

visible_files = []
visible_relative_files = set()
missing_expected = set(expected_relative_files)

# LỖI CŨ: cell dừng ngay khi thấy visible_items > 0, ví dụ mới thấy 4/15 file.
# SỬA: chỉ pass khi thấy đủ file theo drive_shadow_report.
for attempt in range(1, 31):
    visible_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
    visible_relative_files = {
        p.relative_to(archive_source).as_posix()
        for p in visible_files
    }

    if expected_relative_files:
        missing_expected = set(expected_relative_files) - visible_relative_files
        ready = len(missing_expected) == 0
    else:
        missing_expected = set()
        ready = len(visible_files) >= expected_file_count if expected_file_count else len(visible_files) > 0

    print(
        f"- attempt {attempt}/30: "
        f"visible_file_count={len(visible_files)} "
        f"expected_file_count={expected_file_count} "
        f"missing_expected={len(missing_expected)}"
    )

    if ready:
        break

    if missing_expected:
        print("  missing sample:", sorted(missing_expected)[:10])

    time.sleep(10)

if expected_file_count and len(visible_files) < expected_file_count:
    raise RuntimeError(
        f"Drive mount mới thấy {len(visible_files)}/{expected_file_count} file sau drive-shadow. "
        "Không chạy standardize vì sẽ chỉ xử lý một phần dataset. "
        "Đây thường là lỗi sync/cache của Google Drive mount trong Colab. "
        "Hãy đợi thêm vài phút rồi chạy lại riêng cell kiểm tra này."
    )

if expected_relative_files and missing_expected:
    raise RuntimeError(
        f"Drive mount vẫn thiếu {len(missing_expected)} file theo drive-shadow report. "
        f"Ví dụ thiếu: {sorted(missing_expected)[:20]}. "
        "Không chạy standardize khi local mount chưa thấy đủ file."
    )

print("Local Drive mount đã thấy đủ dữ liệu theo drive-shadow report:")
print("- visible_file_count:", len(visible_files))
for p in visible_files[:50]:
    print(" ", p)



Remount Google Drive để Colab nhìn thấy dữ liệu mới...
Mounted at /content/drive
Kiểm tra local archive_source_dir sau drive-shadow:
- archive_source_dir: /content/drive/MyDrive/AIC2026/raw_dataset
- exists: True
Drive mount sync expectation:
- expected_file_count: 15
- expected_relative_files sample: ['Videos_L21_a.zip', 'Videos_L22_a.zip', 'Videos_L23_a.zip', 'Videos_L24_a.zip', 'Videos_L25_a1.zip', 'Videos_L26_a.zip', 'Videos_L26_b.zip', 'Videos_L26_c.zip', 'Videos_L26_d.zip', 'Videos_L26_e.zip', 'Videos_L27_a.zip', 'Videos_L28_a.zip', 'Videos_L29_a.zip', 'Videos_L30_a.zip', 'media-info-aic25-b1.zip']
- attempt 1/30: visible_file_count=15 expected_file_count=15 missing_expected=0
Local Drive mount đã thấy đủ dữ liệu theo drive-shadow report:
- visible_file_count: 15
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
  /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
  /content/drive/MyDrive/AIC2026/r

In [12]:
# BƯỚC 2: Standardize dữ liệu thành raw_videos/metadata.
# Dùng local temp để giảm DriveFS cache và dùng guard/throttle trong package.
# CLI cần được agent sửa để hỗ trợ:
# --resume, --progress-path, --min-free-gb, --drive-sync-sleep-seconds,
# --cleanup-every-files, --cleanup-every-gb.

from pathlib import Path
import json
import shutil
import subprocess

archive_source = Path(config.archive_source_dir)
standardized_root = Path(config.archive_target_dir)
temp_extract = Path(config.temp_extract_dir)
drive_shadow_report_path = Path(config.workspace) / "drive_shadow_report.json"

raw_video_dir = standardized_root / "raw_videos"
metadata_dir = standardized_root / "metadata"
progress_path = standardized_root / "standardize_progress.jsonl"
report_path = standardized_root / "standardize_archives_report.json"

print("BƯỚC 2: Standardize with local temp + DriveFS-safe guards")
print("- archive_source:", archive_source)
print("- standardized_root:", standardized_root)
print("- temp_extract:", temp_extract)
print("- progress_path:", progress_path)
print("- report_path:", report_path)

if not archive_source.exists():
    raise RuntimeError(f"Không thấy archive_source_dir: {archive_source}")

# Với dataset lớn, temp phải nằm ở local runtime để giảm 1 lượt ghi qua Google DriveFS.
# Không dùng /content/drive làm temp vì DriveFS sẽ tạo cache/upload buffer nhiều lần.
temp_extract_str = str(temp_extract.resolve() if temp_extract.exists() else temp_extract)
if temp_extract_str.startswith("/content/drive/"):
    raise RuntimeError(
        f"temp_extract đang nằm trên Google Drive mount: {temp_extract}. "
        "Với dataset lớn, hãy đặt config.temp_extract_dir = '/content/aic_scratch'."
    )

if not temp_extract_str.startswith("/content/"):
    print(
        "WARNING: temp_extract không nằm dưới /content. "
        "Trên Colab nên dùng /content/aic_scratch để cleanup nhanh và tránh DriveFS temp."
    )

standardized_root.mkdir(parents=True, exist_ok=True)

# Cleanup scratch trước khi chạy để không kế thừa rác từ lần run trước.
# Chỉ xóa temp_extract local, không xóa source/output trên Drive.
if temp_extract.exists():
    print("Cleanup local temp_extract trước khi standardize:", temp_extract)
    shutil.rmtree(temp_extract, ignore_errors=True)
temp_extract.mkdir(parents=True, exist_ok=True)

print("Disk trước standardize:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(temp_extract)!r} 2>/dev/null || true", shell=True, check=False)

source_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
print("- source file count:", len(source_files))
for p in source_files[:30]:
    print(" source:", p)

if len(source_files) == 0:
    raise RuntimeError("archive_source_dir đang rỗng. Không chạy standardize.")

# Guard chống lỗi Drive mount chỉ thấy một phần file sau drive-shadow.
# Nếu report nói copy/skip 15 file mà local mount mới thấy 4, dừng ngay tại đây.
if drive_shadow_report_path.exists():
    drive_shadow_report = json.loads(drive_shadow_report_path.read_text(encoding="utf-8"))
    expected_relative_files = sorted({
        str(item.get("path", "")).strip("/")
        for item in drive_shadow_report.get("items", [])
        if item.get("kind") == "file" and item.get("status") in {"copied", "skipped_existing"} and item.get("path")
    })
    expected_file_count = len(expected_relative_files) or (
        int(drive_shadow_report.get("copied_files", 0)) + int(drive_shadow_report.get("skipped_existing", 0))
    )

    if expected_relative_files:
        source_relative_files = {p.relative_to(archive_source).as_posix() for p in source_files}
        missing_expected = sorted(set(expected_relative_files) - source_relative_files)
        if missing_expected:
            raise RuntimeError(
                f"archive_source_dir chỉ thấy {len(source_files)} file nhưng còn thiếu "
                f"{len(missing_expected)} file theo drive-shadow report. "
                f"Ví dụ thiếu: {missing_expected[:20]}. "
                "Không chạy standardize vì sẽ tạo dataset thiếu. "
                "Hãy chạy lại cell remount/kiểm tra Drive cho tới khi đủ file."
            )
    elif expected_file_count and len(source_files) < expected_file_count:
        raise RuntimeError(
            f"archive_source_dir chỉ thấy {len(source_files)}/{expected_file_count} file theo drive-shadow report. "
            "Không chạy standardize vì sẽ tạo dataset thiếu. "
            "Hãy chạy lại cell remount/kiểm tra Drive cho tới khi đủ file."
        )

# Fail sớm nếu package chưa có các option disk-safe mới.
help_result = subprocess.run(
    [sys.executable, "-m", "system1.cli", "standardize-archives", "--help"],
    text=True,
    capture_output=True,
    check=False,
)
help_text = (help_result.stdout or "") + "\n" + (help_result.stderr or "")
required_options = [
    "--min-free-gb",
    "--drive-sync-sleep-seconds",
    "--cleanup-every-files",
    "--cleanup-every-gb",
]
missing_options = [opt for opt in required_options if opt not in help_text]
if missing_options:
    raise RuntimeError(
        "Package hiện tại chưa hỗ trợ các option disk-safe cho standardize-archives: "
        f"{missing_options}. Hãy để agent sửa package trước khi chạy BƯỚC 2."
    )

if config.run_standardize_archives:
    run_cli([
        "standardize-archives",
        "--source-dir", str(archive_source),
        "--target-dir", str(standardized_root),
        "--temp-dir", str(temp_extract),
        "--resume",
        "--progress-path", str(progress_path),
        "--min-free-gb", "15",
        "--drive-sync-sleep-seconds", "30",
        "--cleanup-every-files", "1",
        "--cleanup-every-gb", "50",
    ])
else:
    print("config.run_standardize_archives=False, skip BƯỚC 2.")

print("Disk sau standardize:")
subprocess.run("df -h / /content /content/drive 2>/dev/null || df -h / /content", shell=True, check=False)
subprocess.run(f"du -sh {str(temp_extract)!r} 2>/dev/null || true", shell=True, check=False)

print("Kiểm tra standardized_root:", standardized_root)
if not standardized_root.exists():
    raise FileNotFoundError(f"Không tồn tại archive_target_dir sau standardize: {standardized_root}")

print("- raw_videos exists:", raw_video_dir.exists())
print("- metadata exists:", metadata_dir.exists())
print("- progress exists:", progress_path.exists())
print("- report exists:", report_path.exists())

if not raw_video_dir.exists():
    raise RuntimeError(f"Không tìm thấy raw_videos sau standardize: {raw_video_dir}")
if not metadata_dir.exists():
    raise RuntimeError(f"Không tìm thấy metadata sau standardize: {metadata_dir}")

media_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav"}
media_files = sorted([p for p in raw_video_dir.rglob("*") if p.is_file() and p.suffix.lower() in media_exts])
json_files = sorted([p for p in metadata_dir.rglob("*.json") if p.is_file()])

print("media_count:", len(media_files))
print("metadata_count:", len(json_files))

for p in media_files[:20]:
    print(" media:", p)
for p in json_files[:20]:
    print(" metadata:", p)

if len(media_files) == 0:
    raise RuntimeError("raw_videos tồn tại nhưng không có media file. Cần sửa standardize_archive_source hoặc archive_source_dir sai.")

# Không fail nếu metadata thừa hoặc thiếu metadata ở đây.
# Ingest local sẽ dùng --pairing-policy video-primary:
# - video là nguồn chính
# - metadata thiếu thì tạo minimal metadata
# - metadata thừa thì ignore/report
video_stems = {p.stem for p in media_files}
metadata_stems = {p.stem for p in json_files}
missing_meta = sorted(video_stems - metadata_stems)
unmatched_meta = sorted(metadata_stems - video_stems)

print("pairing quick check:")
print("- videos missing metadata:", len(missing_meta))
print("- metadata without video:", len(unmatched_meta))

if missing_meta:
    print("Ví dụ video thiếu metadata:", missing_meta[:30])
if unmatched_meta:
    print("Ví dụ metadata thừa:", unmatched_meta[:30])

print("Standardize OK enough for tolerant ingest:")
print("- raw_video_dir:", raw_video_dir)
print("- metadata_dir:", metadata_dir)


BƯỚC 2: Standardize with local temp + DriveFS-safe guards
- archive_source: /content/drive/MyDrive/AIC2026/raw_dataset
- standardized_root: /content/drive/MyDrive/AIC2026/standardize
- temp_extract: /content/temp_extract
- progress_path: /content/drive/MyDrive/AIC2026/standardize/standardize_progress.jsonl
- report_path: /content/drive/MyDrive/AIC2026/standardize/standardize_archives_report.json
Disk trước standardize:
- source file count: 15
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L21_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L22_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L23_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L24_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L25_a1.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_a.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_b.zip
 source: /content/drive/MyDrive/AIC2026/raw_dataset/Videos_L26_c.zip


In [17]:
# BƯỚC 2.5: Flush/remount Drive sau khi standardize ghi nhiều file lớn.
# Mục tiêu:
# - tránh lỗi DriveFS "Transport endpoint is not connected" ở BƯỚC 3
# - kiểm tra không chỉ video stat mà cả metadata read_text/json.loads

from google.colab import drive
from pathlib import Path
import time
import os
import subprocess
import json

standardized_root = Path(config.archive_target_dir)
raw_video_dir = standardized_root / "raw_videos"
metadata_dir = standardized_root / "metadata"

print("BƯỚC 2.5: DriveFS settle before ingest")
print("- standardized_root:", standardized_root)
print("- raw_video_dir:", raw_video_dir)
print("- metadata_dir:", metadata_dir)

# Ép OS flush write buffer.
try:
    os.system("sync")
except Exception as exc:
    print("sync warning:", repr(exc))

# Cho DriveFS có thời gian upload/clear cache.
time.sleep(30)

# Remount để tránh endpoint cũ bị gãy sau khi ghi nhiều file lớn.
try:
    drive.flush_and_unmount()
except Exception as exc:
    print("flush_and_unmount warning:", repr(exc))

time.sleep(10)

# Thử lazy unmount nếu endpoint cũ vẫn kẹt.
try:
    subprocess.run(
        "fusermount -u /content/drive 2>/dev/null || true",
        shell=True,
        check=False,
    )
    subprocess.run(
        "umount -l /content/drive 2>/dev/null || true",
        shell=True,
        check=False,
    )
except Exception as exc:
    print("manual unmount warning:", repr(exc))

time.sleep(5)
drive.mount("/content/drive", force_remount=True)

# Re-create Path objects sau remount để tránh giữ object cũ.
standardized_root = Path(config.archive_target_dir)
raw_video_dir = standardized_root / "raw_videos"
metadata_dir = standardized_root / "metadata"

if not standardized_root.exists():
    raise RuntimeError(f"Không thấy standardized_root sau remount: {standardized_root}")

if not raw_video_dir.exists():
    raise RuntimeError(f"Không thấy raw_videos sau remount: {raw_video_dir}")

if not metadata_dir.exists():
    raise RuntimeError(f"Không thấy metadata sau remount: {metadata_dir}")

media_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav"}

media_files = sorted(
    p for p in raw_video_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in media_exts
)

json_files = sorted(
    p for p in metadata_dir.rglob("*.json")
    if p.is_file()
)

print("- media_count after remount:", len(media_files))
print("- metadata_count after remount:", len(json_files))

if not media_files:
    raise RuntimeError("Không thấy media file sau remount. Không chạy ingest.")

if not json_files:
    print("WARNING: Không thấy metadata json nào. Nếu dùng video-primary thì ingest vẫn có thể chạy với metadata generated.")

failed = []
total_media_bytes = 0
total_metadata_bytes = 0

# Probe stat toàn bộ media để phát hiện Drive mount chết trước khi ingest.
for p in media_files:
    try:
        total_media_bytes += p.stat().st_size
    except Exception as exc:
        failed.append(("video_stat", str(p), repr(exc)))
        if len(failed) >= 10:
            break

# Probe read_text + json.loads toàn bộ metadata để bắt lỗi DriveFS khi đọc file JSON.
if not failed:
    for p in json_files:
        try:
            text = p.read_text(encoding="utf-8")
            total_metadata_bytes += len(text.encode("utf-8"))
            json.loads(text)
        except Exception as exc:
            failed.append(("metadata_read", str(p), repr(exc)))
            if len(failed) >= 10:
                break

print("- probed_media_count:", len(media_files))
print("- probed_metadata_count:", len(json_files))
print("- total_media_gb:", round(total_media_bytes / (1024**3), 2))
print("- total_metadata_mb:", round(total_metadata_bytes / (1024**2), 2))

if failed:
    raise RuntimeError(
        "DriveFS chưa ổn định. Không chạy BƯỚC 3. "
        f"Ví dụ lỗi: {failed[:5]}. "
        "Hãy restart runtime hoặc chạy lại BƯỚC 2.5 sau vài phút."
    )

print("DriveFS OK: video stat + metadata read đều pass. Có thể chạy BƯỚC 3 ingest.")

BƯỚC 2.5: DriveFS settle before ingest
- standardized_root: /content/drive/MyDrive/AIC2026/standardize
- raw_video_dir: /content/drive/MyDrive/AIC2026/standardize/raw_videos
- metadata_dir: /content/drive/MyDrive/AIC2026/standardize/metadata
Mounted at /content/drive
- media_count after remount: 834
- metadata_count after remount: 1707
- probed_media_count: 834
- probed_metadata_count: 1707
- total_media_gb: 72.1
- total_metadata_mb: 2.0
DriveFS OK: video stat + metadata read đều pass. Có thể chạy BƯỚC 3 ingest.


In [16]:
# BƯỚC 3: Ingest local trực tiếp từ standardized_root.
# Không upload raw video lên HF trước khi chia batch.
# Ingest local chỉ đọc path + metadata + ffprobe nếu pipeline bật probe.

from pathlib import Path
import os

standardized_root = Path(config.archive_target_dir)
release_root = output_base / config.release_id

print("BƯỚC 3: Ingest local")
print("- standardized_root:", standardized_root)
print("- release_root:", release_root)

if not standardized_root.exists():
    raise RuntimeError(f"Không thấy standardized_root: {standardized_root}")

if not (standardized_root / "raw_videos").exists():
    raise RuntimeError(f"Không thấy raw_videos: {standardized_root / 'raw_videos'}")

if not (standardized_root / "metadata").exists():
    raise RuntimeError(f"Không thấy metadata: {standardized_root / 'metadata'}")

# Source đang nằm trên Google Drive mount nên không chạy nhiều worker.
os.environ["AIC_INGEST_MAX_WORKERS"] = "1"

if config.run_ingest:
    run_cli([
        "ingest",
        "--mode", config.execution_mode,
        "--output", str(output_base),
        "--source-uri", str(standardized_root),
        "--max-workers", "1",
        "--pairing-policy", "video-primary",
        "--no-resume",
    ])
else:
    print("Bỏ qua ingest theo config.")

videos_found = sorted(output_base.rglob("videos.parquet"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet sau ingest local.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet sau ingest local.")

BƯỚC 3: Ingest local
- standardized_root: /content/drive/MyDrive/AIC2026/standardize
- release_root: /content/output/canonical_release_v002

CWD: /content/Multimodal-Agentic-Retrieval-Engine
PYTHONPATH prefix: /content/Multimodal-Agentic-Retrieval-Engine/system1/src
RUN: /usr/bin/python3 -m system1.cli ingest --mode bronze_fast --output /content/output --source-uri /content/drive/MyDrive/AIC2026/standardize --max-workers 1 --pairing-policy video-primary --no-resume
[ingest] source_backend=local source_root=/content/drive/MyDrive/AIC2026/standardize max_workers=1 pairing_policy=video-primary
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/lib/python3.12/pathlib.py:1028 in read_text                             │
│                                                                              │
│   1025 │   │   """                                                           │
│   1026 │   │   encoding = io.text_encoding(encoding)                        

RuntimeError: CLI failed with exit code 1: ingest --mode bronze_fast --output /content/output --source-uri /content/drive/MyDrive/AIC2026/standardize --max-workers 1 --pairing-policy video-primary --no-resume

In [ ]:
# BƯỚC 4: Chia batch từ videos.parquet đã tạo bằng ingest local.

print("BƯỚC 4: Assign batches")
print("- output_base:", output_base)
print("- num_batches:", config.num_batches)

if config.run_assign_batches:
    run_cli([
        "assign-batches",
        "--mode", config.execution_mode,
        "--num-batches", str(config.num_batches),
        "--output", str(output_base),
        "--no-resume",
    ])
else:
    print("Bỏ qua assign-batches theo config.")

batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("batch_manifest.csv found:", batch_manifest_found)
print("batch_*.txt found:", batch_txt_found[:30])

if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv sau assign-batches.")
if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt sau assign-batches.")

In [ ]:
# BƯỚC 5: Gom report phụ vào release manifests.

from pathlib import Path
import shutil

release_root = output_base / config.release_id
manifests_root = release_root / "manifests"
manifests_root.mkdir(parents=True, exist_ok=True)

extra_reports = [
    Path(config.workspace) / "drive_shadow_report.json",
    Path(config.archive_target_dir) / "standardize_archives_report.json",
    Path(config.archive_target_dir) / "standardize_progress.jsonl",
]

for report_path in extra_reports:
    if report_path.exists():
        target = manifests_root / report_path.name
        shutil.copy2(report_path, target)
        print("Copied report:", report_path, "->", target)
    else:
        print("Report not found, skip:", report_path)

print("Release root:", release_root)
print("Release files:")
for p in sorted(release_root.rglob("*"))[:100]:
    print(" ", p)


In [ ]:
# BƯỚC 6: Upload processed artifacts nhỏ lên Hugging Face processed repo.
# Không upload raw_videos ở bước này.

from pathlib import Path
import os

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

release_root = output_base / config.release_id
processed_repo_id = config.hf_release_repo

print("BƯỚC 6: Upload processed artifacts")
print("- run_upload_processed:", config.run_upload_processed)
print("- processed_repo_id:", processed_repo_id)
print("- release_root:", release_root)

if not config.run_upload_processed:
    print("Bỏ qua upload processed artifacts theo config.")
else:
    if not release_root.exists():
        raise RuntimeError(f"Không thấy release_root: {release_root}")

    hf_token = (
        os.environ.get("AIC_HF_TOKEN")
        or os.environ.get("HF_TOKEN")
    )

    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass

    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để upload processed artifacts.")

    api = HfApi(token=hf_token)

    upload_patterns = [
        "tables/videos.parquet",
        "raw_mapping/media_store_manifest.parquet",
        "manifests/dataset_report.json",
        "manifests/ingestion_errors.jsonl",
        "manifests/missing_metadata.json",
        "manifests/unmatched_metadata.json",
        "manifests/batch_manifest.csv",
        "manifests/batch_*.txt",
        "manifests/drive_shadow_report.json",
        "manifests/standardize_archives_report.json",
        "manifests/standardize_progress.jsonl",
    ]

    files_to_upload = []

    for pattern in upload_patterns:
        files_to_upload.extend(sorted(release_root.glob(pattern)))

    # De-duplicate
    files_to_upload = sorted(set(files_to_upload))

    print("- file_count:", len(files_to_upload))

    if not files_to_upload:
        raise RuntimeError("Không có processed artifact nào để upload.")

    for local_path in files_to_upload:
        relative_path = local_path.relative_to(release_root).as_posix()
        remote_path = f"{config.release_id}/{relative_path}"

        print("Uploading:", local_path, "->", remote_path)

        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=remote_path,
            repo_id=processed_repo_id,
            repo_type="dataset",
            token=hf_token,
        )

    print("Uploaded processed artifacts to HF:", processed_repo_id)


In [ ]:
# BƯỚC 7: Upload standardized raw lên HF raw repo có version prefix.
# Dùng CLI command mới: upload-standardized-raw.
# CLI cần được agent thêm trước khi bật config.run_upload_standardized_raw=True.

from pathlib import Path

standardized_root = Path(config.archive_target_dir)

print("BƯỚC 7 OPTIONAL: Upload standardized raw")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- standardized_root:", standardized_root)
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)

if config.run_upload_standardized_raw:
    if not standardized_root.exists():
        raise RuntimeError(f"Không thấy standardized_root: {standardized_root}")
    if not (standardized_root / "raw_videos").exists():
        raise RuntimeError(f"Không thấy raw_videos: {standardized_root / 'raw_videos'}")
    if not (standardized_root / "metadata").exists():
        raise RuntimeError(f"Không thấy metadata: {standardized_root / 'metadata'}")

    run_cli([
        "upload-standardized-raw",
        "--source-dir", str(standardized_root),
        "--target-hf-repo-id", config.hf_canonical_repo,
        "--raw-import-id", config.raw_import_id,
    ])
else:
    print("config.run_upload_standardized_raw=False, skip upload standardized raw.")


In [ ]:
# BƯỚC 8: Cập nhật processed media_store_manifest để trỏ tới HF raw repo versioned.
# Chỉ chạy sau khi BƯỚC 7 upload raw thành công.

from pathlib import Path
import os
import json
import pandas as pd

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

release_root = output_base / config.release_id
media_manifest_path = release_root / "raw_mapping" / "media_store_manifest.parquet"
dataset_report_path = release_root / "manifests" / "dataset_report.json"

raw_repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")

print("BƯỚC 8: Update processed manifest with HF raw refs")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- media_manifest_path:", media_manifest_path)
print("- dataset_report_path:", dataset_report_path)
print("- raw_repo_id:", raw_repo_id)
print("- raw_import_id:", raw_import_id)

if not config.run_upload_standardized_raw:
    print("config.run_upload_standardized_raw=False, skip update media_store_manifest.")
else:
    if not media_manifest_path.exists():
        raise RuntimeError(f"Không thấy media_store_manifest.parquet: {media_manifest_path}")
    if not dataset_report_path.exists():
        raise RuntimeError(f"Không thấy dataset_report.json: {dataset_report_path}")

    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để upload lại processed manifest.")

    media_df = pd.read_parquet(media_manifest_path)

    required_cols = {"video_id", "video_filename", "metadata_filename"}
    missing_cols = sorted(required_cols - set(media_df.columns))
    if missing_cols:
        raise RuntimeError(f"media_store_manifest.parquet thiếu columns: {missing_cols}")

    media_df["canonical_backend"] = "hf_dataset"
    media_df["canonical_repo_id"] = raw_repo_id
    media_df["canonical_repo_type"] = "dataset"
    media_df["canonical_import_id"] = raw_import_id
    media_df["canonical_video_path"] = media_df["video_filename"].map(
        lambda name: f"{raw_import_id}/raw_videos/{name}"
    )
    media_df["canonical_metadata_path"] = media_df["metadata_filename"].map(
        lambda name: f"{raw_import_id}/metadata/{name}"
    )

    media_df.to_parquet(media_manifest_path, index=False)

    dataset_report = json.loads(dataset_report_path.read_text(encoding="utf-8"))
    dataset_report.update({
        "canonical_backend": "hf_dataset",
        "canonical_repo_id": raw_repo_id,
        "canonical_repo_type": "dataset",
        "canonical_import_id": raw_import_id,
        "canonical_manifest": f"{raw_import_id}/manifests/canonical_file_manifest.jsonl",
        "canonical_report": f"{raw_import_id}/manifests/canonical_import_report.json",
    })
    dataset_report_path.write_text(
        json.dumps(dataset_report, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    api = HfApi(token=hf_token)
    processed_repo_id = config.hf_release_repo

    uploads = [
        (media_manifest_path, f"{config.release_id}/raw_mapping/media_store_manifest.parquet"),
        (dataset_report_path, f"{config.release_id}/manifests/dataset_report.json"),
    ]

    for local_path, remote_path in uploads:
        print("Uploading updated processed artifact:", local_path, "->", remote_path)
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=remote_path,
            repo_id=processed_repo_id,
            repo_type="dataset",
            token=hf_token,
        )

    print("Updated processed media_store_manifest and dataset_report.")
    display(media_df.head())


In [ ]:
# BƯỚC 9: Preview output của notebook 00.

from pathlib import Path
import pandas as pd

videos_found = sorted(output_base.rglob("videos.parquet"))
batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("Preview notebook 00 outputs")
print("- videos.parquet found:", videos_found)
print("- media_store_manifest.parquet found:", media_manifest_found)
print("- batch_manifest.csv found:", batch_manifest_found)
print("- batch_*.txt count:", len(batch_txt_found))

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet.")
if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv.")
if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt.")

videos_path = videos_found[0]
media_manifest_path = media_manifest_found[0]
batch_manifest_path = batch_manifest_found[0]

videos_df = pd.read_parquet(videos_path)
media_df = pd.read_parquet(media_manifest_path)
batch_df = pd.read_csv(batch_manifest_path)

print("\nvideos.parquet:", videos_path)
print("video_count:", len(videos_df))
display(videos_df.head())

print("\nmedia_store_manifest.parquet:", media_manifest_path)
print("media_manifest_rows:", len(media_df))
display(media_df.head())

print("\nbatch_manifest.csv:", batch_manifest_path)
print("batch_rows:", len(batch_df))
display(batch_df.head())

print("\nBatch txt files:")
for p in batch_txt_found[:30]:
    print(" ", p)

print("\nNotebook 00 hoàn tất nếu cell này chạy xong.")


In [ ]:
# BƯỚC 10: Kiểm tra cấu trúc HF processed repo.

from pathlib import Path
import os
import json
import pandas as pd

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi, hf_hub_download

repo_id = config.hf_release_repo
release_id = config.release_id

hf_token = (
    os.environ.get("AIC_HF_TOKEN")
    or os.environ.get("HF_TOKEN")
)

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
            os.environ["AIC_HF_TOKEN"] = hf_token
    except Exception:
        pass

if not hf_token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF repo.")

api = HfApi(token=hf_token)

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    token=hf_token,
)

release_files = sorted([
    f for f in files
    if f.startswith(f"{release_id}/")
])

print("HF processed repo:", repo_id)
print("release_id:", release_id)
print("release_file_count:", len(release_files))

for f in release_files:
    print(" ", f)

required_exact = [
    f"{release_id}/tables/videos.parquet",
    f"{release_id}/raw_mapping/media_store_manifest.parquet",
    f"{release_id}/manifests/dataset_report.json",
    f"{release_id}/manifests/ingestion_errors.jsonl",
    f"{release_id}/manifests/missing_metadata.json",
    f"{release_id}/manifests/unmatched_metadata.json",
    f"{release_id}/manifests/batch_manifest.csv",
    f"{release_id}/manifests/drive_shadow_report.json",
    f"{release_id}/manifests/standardize_archives_report.json",
    f"{release_id}/manifests/standardize_progress.jsonl",
]

missing_required = [p for p in required_exact if p not in release_files]

batch_txt_files = sorted([
    p for p in release_files
    if p.startswith(f"{release_id}/manifests/batch_") and p.endswith(".txt")
])

forbidden_patterns = [
    "/raw_videos/",
    "/metadata/",
    "/temp_extract/",
    "member_stage_",
    "member_extract_",
]

forbidden_files = [
    p for p in release_files
    if any(pattern in p for pattern in forbidden_patterns)
    or p.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav", ".zip"))
]

print("\nCHECK REQUIRED:")
print("- missing_required:", missing_required)
print("- batch_txt_count:", len(batch_txt_files))
print("- batch_txt_files:", batch_txt_files)

print("\nCHECK FORBIDDEN:")
print("- forbidden_files:", forbidden_files)

if missing_required:
    raise RuntimeError(f"Thiếu required files trên HF processed repo: {missing_required}")

if len(batch_txt_files) == 0:
    raise RuntimeError("Không thấy manifests/batch_*.txt trên HF processed repo.")

if forbidden_files:
    raise RuntimeError(f"HF processed repo có file không nên upload: {forbidden_files}")

# Kiểm tra nội dung nếu raw upload đã bật: media_store_manifest phải có canonical columns.
if config.run_upload_standardized_raw:
    media_manifest_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f"{release_id}/raw_mapping/media_store_manifest.parquet",
        token=hf_token,
    )
    media_df = pd.read_parquet(media_manifest_local)
    required_canonical_cols = {
        "canonical_backend",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_import_id",
        "canonical_video_path",
        "canonical_metadata_path",
    }
    missing_canonical_cols = sorted(required_canonical_cols - set(media_df.columns))
    print("\nCHECK CANONICAL COLUMNS:")
    print("- missing_canonical_cols:", missing_canonical_cols)
    if missing_canonical_cols:
        raise RuntimeError(f"media_store_manifest thiếu canonical columns: {missing_canonical_cols}")

print("\nHF processed folder structure OK.")


In [ ]:
# BƯỚC 11 OPTIONAL: Kiểm tra HF raw repo versioned.
# Chỉ cần chạy nếu đã bật config.run_upload_standardized_raw=True và BƯỚC 7 đã upload xong.

import os

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

raw_repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")

print("BƯỚC 11 OPTIONAL: Check HF raw repo")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- raw_repo_id:", raw_repo_id)
print("- raw_import_id:", raw_import_id)

if not config.run_upload_standardized_raw:
    print("config.run_upload_standardized_raw=False, skip HF raw repo check.")
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                os.environ["HF_TOKEN"] = hf_token
                os.environ["AIC_HF_TOKEN"] = hf_token
        except Exception:
            pass
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

    api = HfApi(token=hf_token)
    files = sorted(api.list_repo_files(
        repo_id=raw_repo_id,
        repo_type="dataset",
        token=hf_token,
    ))

    prefix = f"{raw_import_id}/"
    prefix_files = [f for f in files if f.startswith(prefix)]

    raw_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/raw_videos/")]
    metadata_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/metadata/")]
    manifest_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/manifests/")]

    print("prefix_file_count:", len(prefix_files))
    print("raw_videos count:", len(raw_files))
    print("metadata count:", len(metadata_files))
    print("manifests:")
    for f in manifest_files:
        print(" ", f)

    required = [
        f"{raw_import_id}/manifests/canonical_file_manifest.jsonl",
        f"{raw_import_id}/manifests/canonical_import_report.json",
    ]

    missing = [p for p in required if p not in files]

    forbidden = [
        f for f in prefix_files
        if "standardize_progress.jsonl" in f
        or "standardize_archives_report.json" in f
        or "drive_shadow_report.json" in f
        or "batch_" in f
        or f.endswith("videos.parquet")
        or f.endswith("media_store_manifest.parquet")
    ]

    print("missing required:", missing)
    print("forbidden files:", forbidden)

    if not raw_files:
        raise RuntimeError("HF raw repo không có raw_videos.")
    if not metadata_files:
        raise RuntimeError("HF raw repo không có metadata.")
    if missing:
        raise RuntimeError(f"HF raw repo thiếu required manifests: {missing}")
    if forbidden:
        raise RuntimeError(f"HF raw repo có file không đúng mục đích: {forbidden}")

    print("HF raw repo versioned structure OK.")
